## Dataset preparation: per-student interaction sequences

This notebook builds the dataset used to train a student knowledge-tracing model.
Each row in the sequence files represents one student, with an `attempts` column
containing the ordered list of that student's exercise interactions.

**Input:** Raw interaction logs (~6.4M attempts from 38k students across 6.9k exercises)
and exercise metadata from the MIAAM dataset.

**Steps:**
1. Filter exercises that lack a screenshot (none in practice, but kept as a safety check).
2. Enrich interactions with `activity_id` from the exercises table.
3. Map string IDs (user, exercise, activity) to compact integers for model input.
4. Group each student's attempts chronologically into a single sequence, retaining
   fields needed at training time: exercise/activity IDs, correctness, timestamp,
   duration, work mode, answer, session, and module name.
5. Shuffle students and split 80/20 into train and validation sets.

**Output** (saved to `raw_data_experiments/data/`):
- `exercises.parquet` — exercise metadata with integer IDs
- `train_sequences.parquet` — 30 816 student sequences
- `val_sequences.parquet` — 7 704 student sequences




In [35]:
import polars as pl
import json
import pathlib
import os
import numpy as np


DATASET = pathlib.Path("../../../MIAAM")
SAVE_FOLDER = pathlib.Path("../data")

interactions = pl.read_parquet(DATASET / "data/maths_data_filtered.parquet")
exercises    = pl.read_parquet(DATASET / "data/maths_exercises_table.parquet")

print(interactions.shape, exercises.shape)

(6481693, 14) (7151, 12)


### Filter exercises without a screenshot

In [36]:
SCREENSHOTS = DATASET / "data/screenshots/compressed"
screenshot_ids = {
    f.stem
    for source_dir in SCREENSHOTS.iterdir()
    for f in source_dir.iterdir()
    if f.suffix == ".png"
}

missing_screenshots = exercises.filter(~pl.col("exercise_id").is_in(list(screenshot_ids)))
exercises    = exercises.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
interactions = interactions.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))

## 2) Build per-student sequences


In [37]:
interactions = interactions.join(
    exercises.select(["exercise_id", "activity_id"]),
    on="exercise_id",
    how="left",
)

user_id_map = (
    interactions.select("user_id")
    .unique()
    .sort("user_id")
    .with_row_index("user_id_int")
)

interactions = interactions.join(user_id_map, on="user_id", how="left")


exercise_id_map = (
    interactions.select("exercise_id")
    .unique()
    .sort("exercise_id")
    .with_row_index("exercise_id_int")
)

interactions = interactions.join(exercise_id_map, on="exercise_id", how="left")


activity_id_map = (
    interactions.select("activity_id")
    .unique()
    .sort("activity_id")
    .with_row_index("activity_id_int")
)

interactions = interactions.join(activity_id_map, on="activity_id", how="left")

### Build per-student sequences.

In [38]:
seq_df = (
    interactions
    .select([
        "user_id_int", "exercise_id_int", "exercise_id",
        "activity_id_int", "activity_id", "data_correct",
        "created_at", "data_duration",
        "work_mode", "data_answer", "session_id", "module_name",  # extras you want in history
    ])
    .with_columns(pl.col("created_at").dt.epoch(time_unit="s").alias("created_at"))
    .sort(["user_id_int", "created_at"])
    .group_by("user_id_int", maintain_order=True)
    .agg(pl.struct([
        "exercise_id_int", "exercise_id", "activity_id_int", "activity_id",
        "data_correct", "created_at", "data_duration",
        "work_mode", "data_answer", "session_id", "module_name",
    ]).alias("attempts"))
)

In [39]:
# The dataframe seq_df has one row per student. In the attempts columns the attempts are collected in a list
seq_df.head(2)

user_id_int,attempts
u32,list[struct[11]]
0,"[{3197,""77a65f22-ef59-4958-8c0d-a48bf52a29b3"",22,""0f54e38b-1a89-40aa-ad52-da52baf76155"",true,1697417014,6280.0,""zpdes"",""[1]"",""am::00008169-3452-462c-9b68-775c3d7274f0::2023-10-16T18:59:10.256000+00:00"",""Nombres et calcul""}, {4699,""b620b175-7758-48b0-b7da-33ba4aecdcb3"",22,""0f54e38b-1a89-40aa-ad52-da52baf76155"",true,1697417022,3825.0,""zpdes"",""[1]"",""am::00008169-3452-462c-9b68-775c3d7274f0::2023-10-16T18:59:10.256000+00:00"",""Nombres et calcul""}, … {1363,""302022f4-94be-4e40-a3ca-b2e0341f4a6b"",217,""9b014385-897b-4474-a1e7-285e0847aa08"",true,1704844996,1973.0,""zpdes"",""[0]"",""am::00008169-3452-462c-9b68-775c3d7274f0::2024-01-10T17:18:17.691000+00:00"",""Nombres et calcul""}]"
1,"[{3706,""8991a32c-eb0e-11ec-8fea-0242ac120002"",278,""cdb2070d-3e38-4dda-8db4-64d32a470ce4"",true,1763424397,19037.0,""adaptive-test"",""{'dropzoneAnswers': [{'id': '1', 'tag': '8', 'state': 'dropzone', 'value': '4', 'tagValue': '4'}, {'id': '3', 'tag': '6', 'state': 'dropzone', 'value': '16', 'tagValue': '16'}, {'id': '5', 'tag': '7', 'state': 'dropzone', 'value': '20 – 1', 'tagValue': '20 – 1'}]}"",""am::0001d9aa-3144-423e-9f13-4d8ad099983a::2025-11-18T09:56:34.484000+00:00"",""Nombres et calcul""}, {2609,""62ebe19b-837b-413d-998f-c2750ce0c677"",217,""9b014385-897b-4474-a1e7-285e0847aa08"",true,1763424464,27040.0,""adaptive-test"",""[1]"",""am::0001d9aa-3144-423e-9f13-4d8ad099983a::2025-11-18T09:56:34.484000+00:00"",""Nombres et calcul""}, … {673,""1a26c14c-11b6-11ed-861d-0242ac120002"",181,""8244f77b-1564-4cf2-951b-573de4106775"",true,1763427215,2350.0,""zpdes"",""[0]"",""am::0001d9aa-3144-423e-9f13-4d8ad099983a::2025-11-18T09:56:34.484000+00:00"",""Nombres et calcul""}]"


In [40]:
print('Example of one attempt:')
print(seq_df['attempts'][0][0])

Example of one attempt:
{'exercise_id_int': 3197, 'exercise_id': '77a65f22-ef59-4958-8c0d-a48bf52a29b3', 'activity_id_int': 22, 'activity_id': '0f54e38b-1a89-40aa-ad52-da52baf76155', 'data_correct': True, 'created_at': 1697417014, 'data_duration': 6280.0, 'work_mode': 'zpdes', 'data_answer': '[1]', 'session_id': 'am::00008169-3452-462c-9b68-775c3d7274f0::2023-10-16T18:59:10.256000+00:00', 'module_name': 'Nombres et calcul'}


In [41]:
# Saving the exercise dataset and the datasets (train and val) representing the sequence of attempts 

exercises.write_parquet(SAVE_FOLDER / "exercises.parquet")

shuffled = seq_df.sample(fraction=1.0, shuffle=True, seed=42)
n_val = int(len(shuffled) * 0.2)

val_df = shuffled[:n_val]
train_df = shuffled[n_val:]

train_df.write_parquet(SAVE_FOLDER / "train_sequences.parquet")
val_df.write_parquet(SAVE_FOLDER / "val_sequences.parquet")

In [42]:
train_df.shape

(30816, 2)

In [43]:
val_df.shape

(7704, 2)